# A10c IndoBERT vs GOLD — Inference-Only Evaluation (GPU)

Mengevaluasi model **A7 IndoBERT** (aspek + polarity) yang sudah dilatih pada
silver terhadap label **human-gold** (`gold.jsonl`) — **tanpa training ulang dan
tanpa re-tune**: memakai kalibrasi beku (temperature + detection thresholds dari
silver validation) dan menerapkannya ke gold test split.

Ini referensi human-gold terpisah, BUKAN membuka ulang locked silver test.
Melengkapi `evaluate-gold-baselines` (keyword/TF-IDF) sehingga ketiga model
dibandingkan pada gold test yang sama.

Prasyarat: notebook `06` (A7 model) + `07` (kalibrasi) sudah ada di Drive,
dan `gold.jsonl` sudah di `SIPATURE/data/annotations/gold/`.


## Step 1 — Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Step 2 — Konfigurasi path & parameter

In [2]:
# ============================================================
# CONFIGURATION CELL — satu-satunya tempat mengubah parameter.
# ============================================================
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive/SIPATURE")
DRIVE_SPLIT_DIR = DRIVE_ROOT / "data" / "splits"
DRIVE_GOLD_DIR = DRIVE_ROOT / "data" / "annotations" / "gold"

# A7 run (hasil notebook 06) + kalibrasi beku (hasil notebook 07).
MODEL_RUN_ID = "20260813-1050_indobert-silver-v1"
CALIBRATION_ID = f"{MODEL_RUN_ID}_calibration-v1"
GOLD_EVAL_ID = f"{MODEL_RUN_ID}_gold-v1"

MODEL_RUN_DIR_DRIVE = DRIVE_ROOT / "runs" / MODEL_RUN_ID
CALIBRATION_DIR_DRIVE = DRIVE_ROOT / "calibration" / CALIBRATION_ID

PROJECT_DIR = Path("/content/hackathon/ml")
GOLD_DIR = PROJECT_DIR / "data" / "annotations" / "gold"
SPLIT_DIR = PROJECT_DIR / "data" / "splits"
MODEL_RUN_DIR = PROJECT_DIR / "runs" / MODEL_RUN_ID
CALIBRATION_DIR = PROJECT_DIR / "calibration" / CALIBRATION_ID
OUTPUT_DIR = PROJECT_DIR / "artifacts" / GOLD_EVAL_ID

DRIVE_OUTPUT_DIR = DRIVE_ROOT / "evaluation" / GOLD_EVAL_ID
DRIVE_METRICS_DIR = DRIVE_ROOT / "metrics"

GOLD_FILE = "gold.jsonl"
SPLIT_FILES = [
    "train_silver_v1.jsonl",
    "validation_silver_v1.jsonl",
    "test_silver_v1.jsonl",
    "split_manifest_silver_v1.json",
]

print("Model run (Drive)  :", MODEL_RUN_DIR_DRIVE)
print("Calibration (Drive):", CALIBRATION_DIR_DRIVE)
print("Gold (Drive)       :", DRIVE_GOLD_DIR / GOLD_FILE)
print("Output (lokal)     :", OUTPUT_DIR)
print("Output (Drive)     :", DRIVE_OUTPUT_DIR)


Model run (Drive)  : /content/drive/MyDrive/SIPATURE/runs/20260813-1050_indobert-silver-v1
Calibration (Drive): /content/drive/MyDrive/SIPATURE/calibration/20260813-1050_indobert-silver-v1_calibration-v1
Gold (Drive)       : /content/drive/MyDrive/SIPATURE/data/annotations/gold/gold.jsonl
Output (lokal)     : /content/hackathon/ml/artifacts/20260813-1050_indobert-silver-v1_gold-v1
Output (Drive)     : /content/drive/MyDrive/SIPATURE/evaluation/20260813-1050_indobert-silver-v1_gold-v1


## Step 3 — Clone repository dari GitHub

In [3]:
from google.colab import userdata
import base64
import os
import shutil
import subprocess

token = userdata.get("GITHUB_TOKEN")
assert token, "GITHUB_TOKEN tidak ditemukan di Colab Secrets"

credentials = f"x-access-token:{token}"
authorization = base64.b64encode(credentials.encode()).decode()

repo_dir = "/content/hackathon"
shutil.rmtree(repo_dir, ignore_errors=True)

environment = os.environ.copy()
environment["GIT_CONFIG_COUNT"] = "1"
environment["GIT_CONFIG_KEY_0"] = "http.extraHeader"
environment["GIT_CONFIG_VALUE_0"] = f"Authorization: Basic {authorization}"

result = subprocess.run(
    ["git", "clone", "https://github.com/jodypangaribuan/hackathon.git", repo_dir],
    env=environment,
    text=True,
    capture_output=True,
)

print("Return code:", result.returncode)
print(result.stdout)
print(result.stderr)

assert result.returncode == 0, "Clone gagal. Periksa izin token GitHub."


Return code: 0

Cloning into '/content/hackathon'...



## Step 4 — Verifikasi commit terbaru (git log)

In [4]:
%cd /content/hackathon/ml
!git log --oneline -3


/content/hackathon/ml
4f5c2a1 (HEAD -> main, origin/main, origin/HEAD) feat: IndoBERT-vs-gold inference-only evaluation (module + CLI + notebook 12)
ba6248f Created using Colab
50d2f82 Created using Colab


## Step 5 — Install dependencies

In [5]:
%cd /content/hackathon/ml
!python -m pip uninstall -y torchvision
!python -m pip install -r requirements-colab.lock.txt
!python -m pip install --no-deps -e .


/content/hackathon/ml
Found existing installation: torchvision 0.26.0+cu128
Uninstalling torchvision-0.26.0+cu128:
  Successfully uninstalled torchvision-0.26.0+cu128
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 10.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 73.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 129.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 MB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 129.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.3/37.3 MB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 67.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## Step 6 — Verifikasi GPU & versi package

In [3]:
import importlib.util
import torch
import transformers

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA tersedia:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("Torchvision ditemukan:", importlib.util.find_spec("torchvision") is not None)

from transformers import BertForSequenceClassification
print("BertForSequenceClassification berhasil diimpor: OK")


PyTorch: 2.7.1+cu126
Transformers: 4.53.2
CUDA tersedia: True
GPU: Tesla T4
Torchvision ditemukan: False
BertForSequenceClassification berhasil diimpor: OK


## Step 7 — Copy gold + split + model + kalibrasi dari Drive

In [4]:
import shutil
from pathlib import Path

# gold
GOLD_DIR.mkdir(parents=True, exist_ok=True)
gold_source = DRIVE_GOLD_DIR / GOLD_FILE
assert gold_source.is_file(), f"gold.jsonl tidak ditemukan di Drive: {gold_source}"
shutil.copy2(gold_source, GOLD_DIR / GOLD_FILE)
print("Disalin:", GOLD_FILE)

# split
SPLIT_DIR.mkdir(parents=True, exist_ok=True)
for filename in SPLIT_FILES:
    source = DRIVE_SPLIT_DIR / filename
    assert source.is_file(), f"Split file tidak ditemukan: {source}"
    shutil.copy2(source, SPLIT_DIR / filename)
    print("Disalin:", filename)

# model run (A7) — seluruh folder
MODEL_RUN_DIR.parent.mkdir(parents=True, exist_ok=True)
if MODEL_RUN_DIR.exists():
    shutil.rmtree(MODEL_RUN_DIR)
shutil.copytree(MODEL_RUN_DIR_DRIVE, MODEL_RUN_DIR)
print("Disalin model run:", MODEL_RUN_DIR.name)

# kalibrasi
CALIBRATION_DIR.parent.mkdir(parents=True, exist_ok=True)
if CALIBRATION_DIR.exists():
    shutil.rmtree(CALIBRATION_DIR)
shutil.copytree(CALIBRATION_DIR_DRIVE, CALIBRATION_DIR)
print("Disalin kalibrasi:", CALIBRATION_DIR.name)


Disalin: gold.jsonl
Disalin: train_silver_v1.jsonl
Disalin: validation_silver_v1.jsonl
Disalin: test_silver_v1.jsonl
Disalin: split_manifest_silver_v1.json
Disalin model run: 20260813-1050_indobert-silver-v1
Disalin kalibrasi: 20260813-1050_indobert-silver-v1_calibration-v1


## Step 8 — Import modul sipature_ml

In [5]:
import sys
from pathlib import Path

source_dir = PROJECT_DIR / "src"
assert source_dir.is_dir(), "Folder source SIPATURE tidak ditemukan."
if str(source_dir) not in sys.path:
    sys.path.insert(0, str(source_dir))

import sipature_ml
print("Modul SIPATURE berhasil dimuat dari:")
print(sipature_ml.__file__)


Modul SIPATURE berhasil dimuat dari:
/content/hackathon/ml/src/sipature_ml/__init__.py


## Step 9 — Jalankan evaluasi IndoBERT vs gold

In [6]:
import torch

from sipature_ml.indobert_gold import run_gold_indobert_evaluation

assert torch.cuda.is_available(), "CUDA GPU tidak tersedia."
assert not OUTPUT_DIR.exists(), f"Output dir sudah ada: {OUTPUT_DIR}"

metrics = run_gold_indobert_evaluation(
    SPLIT_DIR,
    MODEL_RUN_DIR,
    CALIBRATION_DIR,
    GOLD_DIR / GOLD_FILE,
    OUTPUT_DIR,
)

print("IndoBERT (gold) aspect Macro F1:", round(metrics["macro_f1"], 4))
print("IndoBERT (gold) aspect Micro F1:", round(metrics["micro_f1"], 4))
print("IndoBERT (gold) polarity Macro F1:", round(metrics["polarity"]["macro_f1"], 4))
print("Basis:", metrics["evaluation_basis"])


IndoBERT (gold) aspect Macro F1: 0.4848
IndoBERT (gold) aspect Micro F1: 0.4622
IndoBERT (gold) polarity Macro F1: 0.5019
Basis: silver-trained A7 IndoBERT, frozen silver-validation calibration (no retrain/re-tune)


## Step 10 — Tampilkan per-aspect & perbandingan silver

In [7]:
print("Per-aspect F1 (gold test):")
for aspect, info in sorted(metrics["per_aspect"].items()):
    print(f"  {aspect:<22} f1={info['f1']:.4f}  (support {info['support']})")

print("\nPerbandingan aspek (Macro F1):")
print("  IndoBERT silver (locked test) : 0.5247")
print(f"  IndoBERT gold   (inference)   : {metrics['macro_f1']:.4f}")


Per-aspect F1 (gold test):
  access                 f1=0.2632  (support 7)
  cleanliness            f1=0.6061  (support 32)
  comfort                f1=0.3514  (support 101)
  crowding               f1=0.4706  (support 9)
  maintenance            f1=0.7273  (support 14)
  opening_hours          f1=0.0000  (support 1)
  parking                f1=0.8000  (support 20)
  price_transparency     f1=0.5075  (support 28)
  public_facilities      f1=0.4528  (support 21)
  safety                 f1=0.4615  (support 9)
  sanitation             f1=0.4643  (support 19)
  scenery                f1=0.3932  (support 88)
  staff_service          f1=0.4138  (support 11)
  waste                  f1=0.8750  (support 9)

Perbandingan aspek (Macro F1):
  IndoBERT silver (locked test) : 0.5247
  IndoBERT gold   (inference)   : 0.4848


## Step 11 — Copy output ke Drive

In [8]:
import shutil
from pathlib import Path

DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for source in sorted(OUTPUT_DIR.glob("*")):
    if source.is_file():
        shutil.copy2(source, DRIVE_OUTPUT_DIR / source.name)
        print(f"Disalin: {source.name} -> {DRIVE_OUTPUT_DIR}")

# Salin juga ke metrics/ agar bisa dibaca notebook perbandingan.
DRIVE_METRICS_DIR.mkdir(parents=True, exist_ok=True)
metric_file = OUTPUT_DIR / "indobert-gold-v1-test-metrics.json"
if metric_file.is_file():
    shutil.copy2(metric_file, DRIVE_METRICS_DIR / metric_file.name)
    print(f"Disalin: {metric_file.name} -> {DRIVE_METRICS_DIR}")


Disalin: indobert-gold-v1-test-metrics.json -> /content/drive/MyDrive/SIPATURE/evaluation/20260813-1050_indobert-silver-v1_gold-v1
Disalin: manifest.json -> /content/drive/MyDrive/SIPATURE/evaluation/20260813-1050_indobert-silver-v1_gold-v1
Disalin: metrics.json -> /content/drive/MyDrive/SIPATURE/evaluation/20260813-1050_indobert-silver-v1_gold-v1
Disalin: indobert-gold-v1-test-metrics.json -> /content/drive/MyDrive/SIPATURE/metrics


## Step 12 — Run summary (hash & metric)

In [9]:
# ============================================================
# RUN SUMMARY — hash, metric, dan temuan.
# ============================================================
from pathlib import Path
from sipature_ml.manifest import sha256_file

print("REFERENCE LABEL TYPE:", metrics["reference_label_type"])
print("GOLD SHA256         :", metrics["gold_sha256"])
print("A7 RUN ID           :", metrics["a7_run_id"])
print("Aspect Macro F1     :", round(metrics["macro_f1"], 4))
print("Aspect Micro F1     :", round(metrics["micro_f1"], 4))
print("Polarity Macro F1   :", round(metrics["polarity"]["macro_f1"], 4))
print("Severity            :", metrics["severity"]["status"])

print("\nOUTPUT DIR (lokal):", OUTPUT_DIR)
print("OUTPUT DIR (Drive) :", DRIVE_OUTPUT_DIR)
print("\nTEMUAN: IndoBERT silver 0.5247 vs gold (angka di atas).")
print("Ini inference-only (model A7 dilatih silver, tanpa re-tune).")


REFERENCE LABEL TYPE: human_gold
GOLD SHA256         : 2376ede57eeec8b54b6548601621ca1157a1d94dfe4bf2b06007335db1e72aaa
A7 RUN ID           : 20260813-1050_indobert-silver-v1
Aspect Macro F1     : 0.4848
Aspect Micro F1     : 0.4622
Polarity Macro F1   : 0.5019
Severity            : unavailable_no_model

OUTPUT DIR (lokal): /content/hackathon/ml/artifacts/20260813-1050_indobert-silver-v1_gold-v1
OUTPUT DIR (Drive) : /content/drive/MyDrive/SIPATURE/evaluation/20260813-1050_indobert-silver-v1_gold-v1

TEMUAN: IndoBERT silver 0.5247 vs gold (angka di atas).
Ini inference-only (model A7 dilatih silver, tanpa re-tune).
